# Verification tests · run once, send back one file

This notebook re-checks the parts of the paper that a reviewer could question, with the exact protocol of `scripts/synthetic_characterisation.py`.

| Test | Question | Time on a T4 |
|---|---|---|
| **A** | Eight initialisations, 14 000 steps, ground truth 0: which solutions are fit-matched? How much do they disagree? | ≈35–50 min |
| **B** | **Corrected barrier test**: the original code could pair an under-converged solution. Here the pair is chosen only from fit-matched solutions. Also a straight-path profile against the target | ≈1 min |
| **C** | Sensitivity along the solution difference vs 200 random directions, on the corrected pair | ≈2 min |
| **D** *(optional)* | Repeat A–C on ground truths 1–3 for error bars | ≈40 min each |

**How to run:** `Runtime → Change runtime type → T4 GPU`. Run Step 1, then **Runtime → Restart session**, then run every other cell from top to bottom (`Runtime → Run after` from Step 2 works). Keep the tab open. At the end, a file **`verification_results.zip`** downloads: send it back. The last cell also prints a short summary you can paste instead.

## Step 1 · Install and clone (then **Runtime → Restart session**)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install uv
!uv pip -q install --system "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
!uv pip -q install --system "transformers<4.57" "torchaudio==2.6.0" "numpy<2.1" h5py scikit-learn
!git clone -q https://github.com/Ivxn404/semantic-decoding-tribe-inversion /content/repo || (cd /content/repo && git pull -q)
print('INSTALL DONE - now click Runtime > Restart session, then run Step 2 onwards')

## Step 2 · Settings, weights and the frozen model

In [ ]:
import os, sys, json, time, math, warnings
os.environ['PROJECT_ROOT'] = '/content/repo'; os.environ['TRIBE_WEIGHTS'] = '/content/repo/weights'
os.environ['RESULTS_DIR'] = '/content/verify'; os.makedirs('/content/verify', exist_ok=True)
os.chdir('/content/repo/scripts'); sys.path.insert(0, '/content/repo/scripts')
sys.path = [p for p in sys.path if not p.endswith('tribev2')]
warnings.filterwarnings('ignore')
from huggingface_hub import hf_hub_download
for f in ['config.yaml', 'best.ckpt']:
    hf_hub_download('facebook/tribev2', f, local_dir='/content/repo/weights')
import numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from load_tribe import load, make_fwd
from synthetic_characterisation import make_truth, init_x, metrics, DEV
net, _ = load(); net = net.to(DEV); fwd = make_fwd(net)
OUT = '/content/verify'
INITS = ["gauss_0.01", "gauss_0.1", "gauss_1.0", "gauss_5.0", "uniform", "zeros", "truth_perturbed", "truth+noise"]
STEPS = 14000            # the paper's budget for the initialisation study
GTS = [0]                # set to [0, 1, 2, 3] for Test D (adds about 40 min per ground truth)
print('device', DEV, '· ready')

## Test A · Eight initialisations, inverted together

All eight starts are optimised in one batch. The loss is the sum of each start's own mean-squared error, so every start follows exactly the trajectory it would follow alone (Adam is element-wise). A start counts as **fit-matched** if its final error is within 2× of the best, as in the paper.

In [ ]:
def invert_batch(y, x0, steps, tag):
    x = x0.clone().requires_grad_(True); K = x.shape[0]
    opt = torch.optim.Adam([x], lr=5e-2); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    Y = y.expand(K, -1, -1); t0 = time.time()
    for i in range(steps):
        opt.zero_grad(); loss = ((fwd(x) - Y) ** 2).mean(dim=(1, 2)).sum(); loss.backward(); opt.step(); sch.step()
        if i in (99, 999) or (i + 1) % 2000 == 0:
            el = time.time() - t0; print(f'[{tag}] step {i+1}/{steps} · {el/60:.1f} min · about {el/(i+1)*(steps-i-1)/60:.0f} min left', flush=True)
    return x.detach()

def pairwise(X):
    K = X.shape[0]; M = np.eye(K)
    for a in range(K):
        for b in range(a + 1, K):
            M[a, b] = M[b, a] = F.cosine_similarity(X[a:a+1], X[b:b+1], dim=-1).mean().item()
    return M

results = {}
for gt in GTS:
    x = make_truth(seed=gt); y = fwd(x).detach()
    x0 = torch.cat([init_x(k, x) for k in INITS])
    X = invert_batch(y, x0, STEPS, f'gt{gt}')
    torch.save({k: X[i:i+1].cpu() for i, k in enumerate(INITS)}, f'{OUT}/solutions_gt{gt}.pt')
    per = {k: metrics(fwd, X[i:i+1], x) for i, k in enumerate(INITS)}
    best = min(v['mse'] for v in per.values()); matched = [k for k in INITS if per[k]['mse'] <= 2 * best]
    M = pairwise(X); mi = [INITS.index(k) for k in matched]; sub = M[np.ix_(mi, mi)]; off = sub[~np.eye(len(mi), dtype=bool)]
    results[gt] = {'per_init': per, 'best_mse': best, 'matched': matched, 'pairwise_all': M.tolist(),
                   'matched_mean_pair': float(off.mean()), 'matched_min_pair': float(off.min())}
    json.dump(results, open(f'{OUT}/testA.json', 'w'), indent=1)
    print(f'\nground truth {gt}: best fit {best:.3e}; fit-matched {len(matched)}/8: {matched}')
    for k in INITS:
        print(f"  {k:16s} fit {per[k]['mse']:.3e} ({per[k]['mse']/best:5.1f}x best)  recovery {per[k]['cosine']:.3f}  {'matched' if k in matched else 'EXCLUDED'}")
    print(f'  agreement among matched: mean {off.mean():.3f}, min {off.min():.3f}')

## Test B · Corrected barrier test

1. **Diagnosis.** Which pair would the original code have chosen (least similar among *all eight*), and how well is that pair fitted?
2. **Corrected test.** The least similar pair *among fit-matched solutions only*, with the paper's protocol: error against the target at x₁ + s·d versus random directions of the same length (50 of them).
3. **Straight-path profile.** Error against the target at 21 points on the line from x₁ to x₂, for every fit-matched pair. A flat valley (null space) stays at the endpoints' level; a rise in the middle is a ridge.

In [ ]:
B = {}
for gt in GTS:
    x = make_truth(seed=gt); y = fwd(x).detach()
    S = {k: v.to(DEV) for k, v in torch.load(f'{OUT}/solutions_gt{gt}.pt').items()}; R = results[gt]
    mse = lambda z: ((fwd(z) - y) ** 2).mean().item()
    def least_similar(keys):
        best = (None, 2.0)
        for i in range(len(keys)):
            for j in range(i + 1, len(keys)):
                c = F.cosine_similarity(S[keys[i]], S[keys[j]], dim=-1).mean().item()
                if c < best[1]: best = ((keys[i], keys[j]), c)
        return best
    (oa, ob), oc = least_similar(INITS)
    (a, b), c = least_similar(R['matched'])
    out = {'old_rule_pair': [oa, ob], 'old_rule_cos': oc, 'old_rule_fits': [R['per_init'][oa]['mse'], R['per_init'][ob]['mse']],
           'pair': [a, b], 'pair_cos': c, 'pair_fits': [R['per_init'][a]['mse'], R['per_init'][b]['mse']]}
    x1, x2 = S[a], S[b]; d = x2 - x1; g = torch.Generator().manual_seed(7); rows = []
    with torch.no_grad():
        for s in [0.1, 0.25, 0.5, 1.0]:
            along = mse(x1 + s * d); rnd = []
            for _ in range(50):
                r = torch.randn(d.shape, generator=g).to(DEV); r = r / r.norm() * d.norm(); rnd.append(mse(x1 + s * r))
            rows.append({'scale': s, 'mse_along_d': along, 'mse_random_median': float(np.median(rnd)), 'random_over_d': float(np.median(rnd) / along)})
        out['barrier_rows'] = rows
        path = np.linspace(0, 1, 21); prof = {}
        for i, ka in enumerate(R['matched']):
            for kb in R['matched'][i + 1:]:
                e = [mse(S[ka] + s * (S[kb] - S[ka])) for s in path]
                prof[f'{ka}|{kb}'] = {'errors': e, 'peak_over_worse_end': max(e[1:-1]) / max(e[0], e[-1])}
        out['path'] = {'s': path.tolist(), 'pairs': prof}
    B[gt] = out; json.dump(B, open(f'{OUT}/testB.json', 'w'), indent=1)
    print(f'ground truth {gt}')
    print(f"  original rule would pair {oa} & {ob} (cos {oc:.3f}); their fits: {out['old_rule_fits'][0]:.2e}, {out['old_rule_fits'][1]:.2e}")
    print(f"  corrected pair {a} & {b} (cos {c:.3f}); fits {out['pair_fits'][0]:.2e}, {out['pair_fits'][1]:.2e}")
    print('  scale   error along d   error random (median)   random / d')
    for r in rows: print(f"  {r['scale']:5.2f}   {r['mse_along_d']:13.3e}   {r['mse_random_median']:21.3e}   {r['random_over_d']:8.2f}x")
    pk = [v['peak_over_worse_end'] for v in prof.values()]
    print(f'  straight paths between matched pairs: peak / worse endpoint  median {np.median(pk):.2f}x, range {min(pk):.2f}-{max(pk):.2f}x over {len(pk)} pairs')
    plt.figure(figsize=(7, 3.4))
    for kk, v in prof.items(): plt.semilogy(path, v['errors'], alpha=.6)
    plt.xlabel('position on the straight path'); plt.ylabel('error vs target'); plt.title(f'straight paths between fit-matched solutions, gt {gt}'); plt.tight_layout(); plt.show()

## Test C · Sensitivity along the solution difference (paper protocol, 200 random directions)

In [ ]:
Cres = {}
for gt in GTS:
    x = make_truth(seed=gt); S = {k: v.to(DEV) for k, v in torch.load(f'{OUT}/solutions_gt{gt}.pt').items()}
    a, b = B[gt]['pair']; x1 = S[a]; eps = 1e-2 * x1.norm().item(); g = torch.Generator().manual_seed(11)
    with torch.no_grad():
        f1 = fwd(x1)
        def s(v): v = v / v.norm(); return ((fwd(x1 + eps * v) - f1).norm() / eps).item()
        sd = s(S[b] - x1); se = s(x1 - x)
        rnd = np.array([s(torch.randn(x1.shape, generator=g).to(DEV)) for _ in range(200)])
        others = {f'{ka}|{kb}': s(S[kb] - S[ka]) for i, ka in enumerate(results[gt]['matched']) for kb in results[gt]['matched'][i + 1:]}
    Cres[gt] = {'pair': [a, b], 'sens_d': sd, 'sens_error': se, 'random_median': float(np.median(rnd)), 'random_min': float(rnd.min()), 'random_max': float(rnd.max()),
                'ratio_median_over_d': float(np.median(rnd) / sd), 'percentile_of_d': float((rnd < sd).mean() * 100),
                'ratio_median_over_error': float(np.median(rnd) / se), 'all_matched_pairs_ratio': {k: float(np.median(rnd) / v) for k, v in others.items()}}
    json.dump(Cres, open(f'{OUT}/testC.json', 'w'), indent=1)
    r = Cres[gt]; print(f"gt {gt}: random median / along d = {r['ratio_median_over_d']:.2f}x (d at percentile {r['percentile_of_d']:.0f}); along the true error {r['ratio_median_over_error']:.2f}x")
    print('  all matched pairs:', ', '.join(f'{v:.2f}x' for v in r['all_matched_pairs_ratio'].values()))

## Summary and download · send `verification_results.zip` back (or paste the printed summary)

In [ ]:
summary = {'steps': STEPS, 'ground_truths': GTS}
for gt in GTS:
    summary[f'gt{gt}'] = {'matched': results[gt]['matched'], 'best_mse': results[gt]['best_mse'], 'agreement_mean': results[gt]['matched_mean_pair'],
        'agreement_min': results[gt]['matched_min_pair'], 'old_rule_pair': B[gt]['old_rule_pair'], 'old_rule_fits': B[gt]['old_rule_fits'],
        'pair': B[gt]['pair'], 'barrier_random_over_d': [round(r['random_over_d'], 3) for r in B[gt]['barrier_rows']],
        'path_peak_over_end_median': float(np.median([v['peak_over_worse_end'] for v in B[gt]['path']['pairs'].values()])),
        'sensitivity_ratio': Cres[gt]['ratio_median_over_d'], 'sensitivity_ratio_error': Cres[gt]['ratio_median_over_error']}
json.dump(summary, open(f'{OUT}/summary.json', 'w'), indent=1)
print(json.dumps(summary, indent=1))
import shutil; shutil.make_archive('/content/verification_results', 'zip', OUT)
from google.colab import files; files.download('/content/verification_results.zip')